In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

# 1. Load dataset augmentasi
df = pd.read_csv('synthetic_15k_complete_final.csv')

# 2. Konfigurasi Fitur State (12 Fitur agar sesuai dengan Masking Layer)
state_features = [
    'rx_mbps_p1', 'util_rx_pct_p1', 'delay_ms_p1', 'policing_rate_kbps_p1',
    'rx_mbps_p2', 'util_rx_pct_p2', 'delay_ms_p2', 'policing_rate_kbps_p2',
    'rx_mbps_p4', 'util_rx_pct_p4', 'delay_ms_p4', 'policing_rate_kbps_p4'
]

# SLA Targets
SLA_DELAY_P1 = 12.0
SLA_DELAY_P2 = 10.0
SLA_DELAY_P4 = 3.5
TARGET_THROUGHPUT_P2 = 15.0

def sigmoid_penalty(val, threshold, k=1.5):
    return 1 / (1 + np.exp(-k * (val - threshold)))

def calculate_sdh_reward(row):
    # Penalti Delay (SLA)
    penalty_p4 = 10 * sigmoid_penalty(row['delay_ms_p4'], threshold=SLA_DELAY_P4)
    penalty_p1 = 5 * sigmoid_penalty(row['delay_ms_p1'], threshold=SLA_DELAY_P1)
    penalty_p2 = 5 * sigmoid_penalty(row['delay_ms_p2'], threshold=SLA_DELAY_P2)
    
    # Incentive Throughput Port 2
    r_throughput_p2 = 5 * np.clip(row['rx_mbps_p2'] / TARGET_THROUGHPUT_P2, 0, 1)
    
    # Penalti Packet Drop
    total_drop = row.get('drop_p1', 0) + row.get('drop_p2', 0) + row.get('drop_p4', 0)
    p_drop = np.clip(total_drop, 0, 5)
    
    return r_throughput_p2 - penalty_p4 - penalty_p1 - penalty_p2 - p_drop

# 3. Normalisasi State
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df[state_features]), columns=state_features)

drl_data = []
for i in range(len(df) - 1):
    state = df_scaled.iloc[i].values
    next_state = df_scaled.iloc[i+1].values
    
    # Action (Berdasarkan perubahan P4)
    rate_t = df.iloc[i]['policing_rate_kbps_p4']
    rate_t_plus_1 = df.iloc[i+1]['policing_rate_kbps_p4']
    action_cont = (rate_t_plus_1 - rate_t) / rate_t
    
    # Action Discrete
    if action_cont > 0.05: action_disc = 1
    elif action_cont < -0.05: action_disc = 2
    else: action_disc = 0
    
    reward = calculate_sdh_reward(df.iloc[i+1])
    
    drl_data.append({
        'state': state.tolist(),
        'action_continuous': action_cont,
        'action_discrete': action_disc,
        'reward': reward,
        'next_state': next_state.tolist(),
        # TAMBAHKAN KOLOM INI UNTUK VALIDASI DI ALLMODEL
        'raw_delay_p1': df.iloc[i+1]['delay_ms_p1'],
        'raw_delay_p2': df.iloc[i+1]['delay_ms_p2'],
        'raw_delay_p4': df.iloc[i+1]['delay_ms_p4']
    })

# Simpan dataset
df_drl_ready = pd.DataFrame(drl_data)
df_drl_ready.to_csv('drl_preprocessed_final.csv', index=False)
joblib.dump(scaler, 'master_scaler.pkl')

print("Preprocessing Selesai! File 'drl_preprocessed_final.csv' sekarang menyertakan kolom raw_delay.")

Preprocessing Selesai! File 'drl_preprocessed_final.csv' sekarang menyertakan kolom raw_delay.
